# 🎬 Movie Recommendation System
### Content-Based Filtering using TF-IDF and Cosine Similarity

This notebook builds a **content-based movie recommender system**. Given a movie title, the system suggests similar movies based on their **overview, genres, and tagline** — without needing any user rating history (which makes it robust to the "cold-start" problem).

**Pipeline overview:**
1. Load and explore the raw movie metadata
2. Clean the data and select relevant columns
3. Engineer a combined text feature (`tags`)
4. Preprocess text (lowercase, remove punctuation/stopwords, lemmatize)
5. Convert text to numerical vectors using **TF-IDF**
6. Compute pairwise **cosine similarity** between movies
7. Build a `recommend()` function to retrieve similar titles
8. Persist the trained artifacts with `pickle` for later reuse (e.g. in an app)

**Dataset:** `movies_metadata.csv`

## 1. Import Libraries

In [ ]:
# Core data handling
import numpy as np
import pandas as pd

# Visualization (available for optional EDA plots)
import seaborn as sns
import matplotlib.pyplot as plt

# Text preprocessing
import ast
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Feature extraction & similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Model persistence
import pickle

# Suppress non-critical warnings for cleaner notebook output
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Download NLTK resources required for stopword removal and lemmatization
# (only needs to run once per environment)
nltk.download('stopwords')
nltk.download('wordnet')

## 2. Load the Dataset

In [ ]:
# Load the raw movie metadata into a DataFrame
df = pd.read_csv('movies_metadata.csv')

## 3. Exploratory Data Analysis (EDA)
Quick look at the raw data before any cleaning is applied.

In [ ]:
# Preview the first 5 rows
df.head()

In [ ]:
# List all available columns
df.columns

In [ ]:
# Check data types and non-null counts for each column
df.info()

In [ ]:
# Dataset dimensions: (rows, columns)
df.shape

In [ ]:
# Count missing values per column
df.isnull().sum()

## 4. Data Cleaning
- Remove duplicate rows
- Keep only the columns relevant to content-based filtering
- Handle missing values

In [ ]:
# Remove duplicate rows and reset the index
df = df.drop_duplicates().reset_index(drop=True)

In [ ]:
# Keep only the columns needed to build the 'tags' feature
df = df[['title', 'overview', 'genres', 'tagline', 'vote_average', 'popularity']]

In [ ]:
# Inspect the trimmed DataFrame
df

In [ ]:
# Re-check missing values after trimming columns
df.isnull().sum()

In [ ]:
# Drop rows with a missing title - a title is required to identify a movie
df = df.dropna(subset=['title'])

In [ ]:
# Fill missing overviews with an empty string so text concatenation doesn't fail later
df['overview'] = df['overview'].fillna('')

## 5. Feature Engineering — Building the `tags` Column
The `genres` column is stored as a stringified list of dictionaries, e.g. `"[{'id': 28, 'name': 'Action'}]"`. We parse it and extract just the genre names, then combine `overview`, `genres`, and `tagline` into one text feature that represents each movie's content.

In [ ]:
# Inspect the raw (stringified) format of the genres column
df.iloc[0]['genres']

In [ ]:
# Parse the stringified list of genre dictionaries and join the genre names into one string
# e.g. "[{'id': 28, 'name': 'Action'}]" -> "Action"
df['genres'] = df['genres'].apply(
    lambda x: " ".join([i['name'] for i in ast.literal_eval(x)])
)

In [ ]:
# Confirm genres were successfully converted to plain text
df.head()

In [ ]:
# Fill missing taglines with an empty string
df['tagline'] = df['tagline'].fillna('')

In [ ]:
# Final missing-value check before combining the text fields
df.isnull().sum()

In [ ]:
# Combine overview, genres, and tagline into a single text feature.
# This 'tags' column captures the semantic content used for similarity matching.
df['tags'] = df['overview'] + " " + df['genres'] + " " + df['tagline']

In [ ]:
# Preview the combined tags column
df.head()

In [ ]:
# Inspect a single tags entry
df['tags'][1]

## 6. Text Preprocessing (NLP)
Clean the `tags` text before vectorization: lowercase, strip punctuation/numbers, remove stopwords, and lemmatize each word to its base form.

In [ ]:
# Initialize the stopword set and lemmatizer once, so they're reused for every row
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

In [ ]:
def preprocess_text(text):
    """
    Clean and normalize a text string for TF-IDF vectorization.

    Steps
    -----
    1. Lowercase the text
    2. Remove punctuation and numbers
    3. Remove common English stopwords (e.g. "the", "is", "and")
    4. Lemmatize each word to its base form (e.g. "running" -> "run")

    Parameters
    ----------
    text : str
        Raw input text.

    Returns
    -------
    str
        Cleaned, space-joined text.
    """
    # Lowercase
    text = str(text).lower()

    # Remove punctuation and non-alphabetic characters
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # Tokenize by whitespace
    words = text.split()

    # Remove stopwords
    words = [word for word in words if word not in stop_words]

    # Lemmatize each remaining word
    words = [lemmatizer.lemmatize(word) for word in words]

    return " ".join(words)

In [ ]:
# Apply the cleaning function to every row of the tags column
df['tags'] = df['tags'].apply(preprocess_text)

In [ ]:
# Reset the index so it lines up with the TF-IDF matrix rows built below
df = df.reset_index(drop=True)

## 7. Build a Title-to-Index Lookup
To retrieve a movie's vector quickly inside `recommend()`, map each title to its row index.

In [ ]:
# Map each movie title to its DataFrame index for fast lookup
indices = pd.Series(df.index, index=df['title']).drop_duplicates()
indices

## 8. Feature Extraction — TF-IDF Vectorization
Convert the cleaned text into numerical vectors using **Term Frequency–Inverse Document Frequency (TF-IDF)**, which weighs words by how distinctive they are to a given movie relative to the whole corpus.

In [ ]:
# max_features   : cap the vocabulary at the 50,000 most informative terms
# ngram_range     : capture both single words and two-word phrases (1,2)
# stop_words      : remove common English stopwords again at the vectorizer level
tfidf = TfidfVectorizer(max_features=50000, ngram_range=(1, 2), stop_words='english')

In [ ]:
# Fit the vectorizer on the corpus and transform 'tags' into a TF-IDF sparse matrix
tfidf_matrix = tfidf.fit_transform(df['tags'])

In [ ]:
# Inspect the resulting sparse matrix (shape: n_movies x vocabulary_size)
tfidf_matrix

## 9. Build the Recommendation Function
We use **cosine similarity** to measure how close two movies are in TF-IDF space, then return the top-N most similar titles for a given movie.

In [ ]:
def recommend(title, n=10):
    """
    Recommend movies similar to a given title using content-based filtering.

    Parameters
    ----------
    title : str
        The movie title to find recommendations for.
    n : int, default=10
        Number of similar movies to return.

    Returns
    -------
    pandas.Series or list
        Titles of the top-n most similar movies, or a message if the
        title is not found in the dataset.
    """
    if title not in indices:
        return ['Movie not found']

    idx = indices[title]

    # Compute similarity between this movie and every other movie
    sim_score = cosine_similarity(tfidf_matrix[idx], tfidf_matrix).flatten()

    # Sort by similarity (descending) and skip index 0 (the movie itself)
    similar_idx = sim_score.argsort()[::-1][1:n + 1]

    return df['title'].iloc[similar_idx]

## 10. Test the Recommender

In [ ]:
# Example: get the 5 movies most similar to 'Avatar'
recommend('Avatar', 5)

## 11. Save Model Artifacts
Persist the fitted TF-IDF matrix, vectorizer, title index, and cleaned DataFrame so they can be reloaded later without re-running the full pipeline (e.g. inside a web app backend).

In [ ]:
# Save the TF-IDF matrix (every movie's content vector)
pickle.dump(tfidf_matrix, open('tfidf_matrix.pkl', 'wb'))

# Save the title -> index lookup
pickle.dump(indices, open('indices.pkl', 'wb'))

# Save the cleaned DataFrame
df.to_pickle('df.pkl')

# Save the fitted TF-IDF vectorizer (needed to transform any new queries later)
pickle.dump(tfidf, open('tfidf.pkl', 'wb'))

## 12. Conclusion & Next Steps

**What we built:** a content-based movie recommender that suggests similar titles using only each movie's overview, genres, and tagline — no user rating history required.

**Possible improvements:**
- Add cast, crew, director, and keyword data for richer, more accurate features
- Blend in collaborative filtering (user ratings) for a hybrid recommender
- Replace TF-IDF with semantic embeddings (Word2Vec, Sentence-BERT) to capture meaning beyond exact word overlap
- Wrap `recommend()` in a simple Flask or Streamlit app for interactive use
- Add evaluation metrics (e.g. precision@k) if user feedback/rating data becomes available